# Advanced (Optional): Approximate Planning by Random Rollout and Monte Carlo Tree Search
Value iteration and policy iteration compute the exact optimal value function $V^{\star}$ and policy $\pi^{\star}$, but they require the full transition model $P(s^{\prime}\mid s,a)$ and reward $R(s,a)$ for every state and action, and they sweep every state to convergence. When the state space is too large for full sweeps, or when the environment is available only as a simulator that can be stepped forward, an alternative is to plan approximately from simulated experience. This reading develops the theory and pseudocode for two such planners: __random rollout__, which estimates the value of a single state by averaging simulated returns, and __Monte Carlo tree search (MCTS)__, which chooses an action at a single root state by building a small search tree guided by rollouts. The companion Watch-Demo and Codio activity notebooks run both planners on the $5\times5$ grid world and compare them against the exact solution from value iteration.

> __Learning Objectives.__
>
> By the end of this reading, you will be able to define and demonstrate mastery of the following key concepts:

> * __Estimate value by random rollout:__ Estimate $V^{\pi}(s_{0})$ by averaging $N$ simulated truncated returns, and state the two error sources of the estimate: Monte Carlo noise with standard error $\sigma_{G}/\sqrt{N}$ and bias from truncating trajectories at horizon $H$.
> * __Improve a policy by rollout lookahead:__ Combine a one-step Bellman backup with rollout-estimated continuation values to compute an improved action at a state, without solving the full MDP.
> * __Describe Monte Carlo tree search:__ Describe the four MCTS phases (selection, expansion, simulation, backpropagation), state the UCT rule that balances exploration and exploitation, and identify the action MCTS returns at the root.

By the end you will be able to read and run the rollout and MCTS notebooks knowing exactly what each algorithm computes. Let's get started!
___


## Estimating Value by Random Rollout
For a fixed policy $\pi:\mathcal{S}\rightarrow\mathcal{A}$ acting in a finite MDP $(\mathcal{S},\mathcal{A},P,R,\gamma)$, the state-value function $V^{\pi}:\mathcal{S}\rightarrow\mathbb{R}$ is the expected discounted return starting from $s$ and following $\pi$ thereafter:
$$
V^{\pi}(s) = \mathbb{E}\left[\left.\sum_{t=0}^{\infty}\gamma^{t}R(s_{t},\pi(s_{t}))\;\right|\;s_{0}=s\right],\quad\gamma\in[0,1).
$$
__Rollout__ estimates $V^{\pi}(s_{0})$ from simulated experience instead of an exact sweep. Starting from $s_{0}$, simulate a trajectory of finite horizon $H$ under $\pi$, sampling each next state from $P(\cdot\mid s_{t},\pi(s_{t}))$, and truncate its discounted return at $H$ steps:
$$
G = \sum_{t=0}^{H-1}\gamma^{t}R(s_{t},\pi(s_{t})).
$$
Repeating this $N$ times independently and averaging gives the __rollout value estimate__:
$$
\hat V^{\pi}(s_{0}) = \frac{1}{N}\sum_{i=1}^{N}G^{(i)},
$$
where $G^{(i)}$ is the return of the $i$-th trajectory. By the law of large numbers, $\hat V^{\pi}(s_{0})\rightarrow\mathbb{E}\left[G\right]$ as $N\rightarrow\infty$, and $\mathbb{E}\left[G\right]\rightarrow V^{\pi}(s_{0})$ as $H\rightarrow\infty$. The estimate therefore carries two separate errors, one controlled by each parameter:

* __Monte Carlo noise (variance), controlled by $N$:__ the standard error of the estimate is
$$
\mathrm{SE}\!\left[\hat V^{\pi}(s_{0})\right] = \frac{\sigma_{G}}{\sqrt{N}},
$$
where $\sigma_{G}$ is the standard deviation of a single trajectory return $G$. Accuracy improves as $N$ grows, but only at rate $1/\sqrt{N}$, so quadrupling $N$ only halves the standard error.
* __Horizon truncation (bias), controlled by $H$:__ if a trajectory has not reached an absorbing state within $H$ steps, the truncated return $G$ never includes the terminal reward, so it undercounts the return whenever that terminal reward is positive. Increasing $H$ shrinks this bias; increasing $N$ only shrinks the estimator's spread around the $H$-dependent quantity $\mathbb{E}\left[G\right]$ — it does not fix a horizon that is too short.

### Algorithm
__Initialize:__ a simulator of the MDP $(\mathcal{S},\mathcal{A},P,R,\gamma)$ that, given a state-action pair $(s,a)$, returns the reward $R(s,a)$ and a sample $s^{\prime}\sim P(\cdot\mid s,a)$; a policy $\pi:\mathcal{S}\rightarrow\mathcal{A}$; a start state $s_{0}\in\mathcal{S}$; a horizon $H\geq 1$; and a trajectory count $N\geq 1$.

For $i = 1,\dots,N$ __do:__
1. Set $s\gets s_{0}$ and $G^{(i)}\gets 0$.
2. For $t = 0,\dots,H-1$ __do:__
   - Choose the action $a\gets\pi(s)$ (for a stochastic base policy, sample $a\sim\pi(\cdot\mid s)$).
   - Accumulate the discounted reward: $G^{(i)}\gets G^{(i)} + \gamma^{t}\,R(s,a)$.
   - Sample the next state $s^{\prime}\sim P(\cdot\mid s,a)$ and set $s\gets s^{\prime}$.

Return the rollout estimate $\hat V^{\pi}(s_{0})\gets\frac{1}{N}\sum_{i=1}^{N}G^{(i)}$.

Every trajectory runs for exactly $H$ steps: absorbing states in this module's MDPs self-loop with zero reward, so a trajectory that reaches the goal simply accumulates zero for its remaining steps. In the module code, `simulate_return` implements one pass of the outer loop (steps 1–2, a single trajectory return) and `rollout_value` implements the full algorithm.
___


## Improving a Policy by Rollout Lookahead
Rollout also supports one-step __policy improvement__. Given a base policy $\pi$ and a state $s$, score each action by an exact one-step Bellman backup with rollout-estimated continuation values:
$$
\hat Q(s,a) = R(s,a) + \gamma\sum_{s^{\prime}\in\mathcal{S}}P(s^{\prime}\mid s,a)\,\hat V^{\pi}(s^{\prime}),
\qquad
\hat\pi(s) = \arg\max_{a\in\mathcal{A}_{s}}\hat Q(s,a),
$$
where each $\hat V^{\pi}(s^{\prime})$ is a rollout estimate of the base policy's value from the successor state $s^{\prime}$. Even when $\pi$ is a poor policy (e.g., uniformly random), $\hat\pi(s)$ typically improves on $\pi(s)$: the exact one-step lookahead corrects the immediate decision, and rollout only needs to approximate the value of "following $\pi$ from here on," not the value of the full MDP. This step requires the one-step model at $s$ — the reward $R(s,a)$ and the transition probabilities $P(s^{\prime}\mid s,a)$ of the immediate successors — but only a simulator beyond it.

### Algorithm
__Initialize:__ the one-step model at state $s$ (the reward $R(s,a)$ and the transition probabilities $P(s^{\prime}\mid s,a)$ for each action $a\in\mathcal{A}_{s}$); a simulator for rollouts; a base policy $\pi:\mathcal{S}\rightarrow\mathcal{A}$; a rollout horizon $H\geq 1$; and a trajectory count $N\geq 1$.

For each action $a\in\mathcal{A}_{s}$ __do:__
1. Identify the successor set $\mathcal{S}_{a}\gets\{s^{\prime}\in\mathcal{S}\,:\,P(s^{\prime}\mid s,a)>0\}$.
2. For each $s^{\prime}\in\mathcal{S}_{a}$, estimate the continuation value $\hat V^{\pi}(s^{\prime})$ with the rollout algorithm above (policy $\pi$, horizon $H$, $N$ trajectories).
3. Compute the lookahead score: $\hat Q(s,a)\gets R(s,a) + \gamma\sum_{s^{\prime}\in\mathcal{S}_{a}}P(s^{\prime}\mid s,a)\,\hat V^{\pi}(s^{\prime})$.

Return the improved action $\hat\pi(s)\gets\arg\max_{a\in\mathcal{A}_{s}}\hat Q(s,a)$.

The Watch-Demo composes this algorithm directly from `rollout_value` and a one-step backup over the transition array; there is no separate library routine.
___


## Monte Carlo Tree Search
Rollout evaluates a *given* policy at a state. __Monte Carlo tree search (MCTS)__ goes further: it chooses a good *action* at a single root state $s_{0}$ by building a small, biased search tree guided by random rollouts, instead of solving the Bellman equations for every state. The tree stores three statistics for each state it has expanded: the visit count $N(s)$ of state $s$, the visit count $N(s,a)$ of action $a$ at $s$, and the mean-return estimate $Q(s,a)$. Each search iteration runs four phases:

1. __Selection.__ Starting at the root, while the current state is already in the tree, choose the action that maximizes the UCT (Upper Confidence bound for Trees) score
$$
\mathrm{UCT}(s,a) = Q(s,a) + c\sqrt{\frac{\ln N(s)}{N(s,a)}},
$$
where $c>0$ is an exploration constant. The first term favors actions with high estimated value (exploitation); the second term grows for actions tried rarely relative to $N(s)$ (exploration). Sample the next state from $P(\cdot\mid s,a)$ and descend until the walk leaves the tree or reaches a depth cap $D$.
2. __Expansion.__ When the walk reaches a state not yet in the tree, add it: initialize $N(s)\gets 0$, and $N(s,a)\gets 0$, $Q(s,a)\gets 0$ for every $a\in\mathcal{A}$.
3. __Simulation (rollout).__ Estimate the value of the newly added state with a random rollout — a single trajectory of $H$ steps under the uniform-random policy (the rollout algorithm above with $N=1$) — rather than by expanding the tree further.
4. __Backpropagation.__ Walk the selection path backward from the leaf to the root. At each state-action pair on the path, the backed-up value is the discounted return observed from that pair onward, $G\gets R(s,a)+\gamma\,G$, and the statistics are updated by an incremental mean: $N(s,a)\gets N(s,a)+1$ and $Q(s,a)\gets Q(s,a)+\left(G-Q(s,a)\right)/N(s,a)$.

After $K$ iterations the tree holds sample-based estimates $Q(s_{0},a)$ for every action at the root. MCTS returns $\arg\max_{a\in\mathcal{A}}Q(s_{0},a)$ — the single best action at the root — and discards the rest of the tree. The returned root action converges to the optimal action $\pi^{\star}(s_{0})$ as the iteration budget grows (Kocsis & Szepesvári, 2006).

### Algorithm
__Initialize:__ a simulator of the MDP $(\mathcal{S},\mathcal{A},P,R,\gamma)$; the root state $s_{0}\in\mathcal{S}$; an iteration budget $K\geq 1$; an exploration constant $c>0$; a rollout horizon $H\geq 1$; a depth cap $D\geq 1$; and an empty tree (no states expanded).

For $k = 1,\dots,K$ __do:__
1. __Select.__ Set $s\gets s_{0}$, $d\gets D$, and the path $\mathcal{P}\gets()$ (an empty sequence). While $s$ is in the tree and $d>0$ __do:__
   - Increment the state count: $N(s)\gets N(s)+1$.
   - Choose $a\gets\arg\max_{a\in\mathcal{A}}\left(Q(s,a) + c\sqrt{\dfrac{\ln\left(N(s)+1\right)}{N(s,a)+1}}\right)$.
   - Append $(s,a)$ to $\mathcal{P}$, sample $s^{\prime}\sim P(\cdot\mid s,a)$, and set $s\gets s^{\prime}$ and $d\gets d-1$.
2. __Expand.__ If $d>0$ (the walk stopped at a state not in the tree), add $s$ to the tree: $N(s)\gets 0$, and $N(s,a)\gets 0$, $Q(s,a)\gets 0$ for every $a\in\mathcal{A}$.
3. __Simulate.__ If $d>0$, set $G\gets$ the discounted return of one random rollout of $H$ steps from $s$ (the rollout algorithm above with $N=1$ and the uniform-random policy); if $d=0$ (the depth cap was reached), set $G\gets 0$.
4. __Backpropagate.__ For each $(s,a)\in\mathcal{P}$ from last to first __do:__ set $G\gets R(s,a)+\gamma\,G$, $N(s,a)\gets N(s,a)+1$, and $Q(s,a)\gets Q(s,a)+\left(G-Q(s,a)\right)/N(s,a)$.

Return the root action $\arg\max_{a\in\mathcal{A}}Q(s_{0},a)$.

The $+1$ offsets inside the UCT score keep the logarithm and the ratio defined for states and actions that have not yet been visited; this smoothed form is what the module's `mcts` routine implements, with $K=$ `iterations`, $H=$ `horizon`, and $D=$ `depth` set by `MyMCTSModel`, along with the exploration constant `c`. Tree growth is capped at $D$ moves from the root: a walk that reaches the cap backs up only the rewards collected along its path.
___


## Choosing a Planner
Value iteration, rollout, and MCTS answer different questions with different requirements. Exact dynamic programming computes the value of *every* state; rollout estimates the value of *one* state under a *given* policy; MCTS returns a good *action* at *one* state.

| | Value iteration / policy iteration | Random rollout | Monte Carlo tree search |
| :--- | :--- | :--- | :--- |
| Requires | full model: $P(s^{\prime}\mid s,a)$ and $R(s,a)$ for every state-action pair | a simulator and a policy to evaluate (one-step improvement also needs the transition probabilities at the current state) | a simulator |
| Computes | $V^{\star}$ and $\pi^{\star}$ for every state | $\hat V^{\pi}(s_{0})$ at one state; an improved action with one-step lookahead | the estimated best action at the root $s_{0}$ |
| Error | within tolerance $\epsilon$ of $V^{\star}$ (contraction) | Monte Carlo noise with standard error $\sigma_{G}/\sqrt{N}$, plus horizon-truncation bias | root action converges to $\pi^{\star}(s_{0})$ as the iteration budget $K$ grows |
| Simulation cost | none (full sweeps, $O(\lvert\mathcal{S}\rvert^{2}\lvert\mathcal{A}\rvert)$ per sweep) | $N\times H$ simulated steps per estimate | at most $K\times(D+H)$ simulated steps per action choice |

When the full model fits in memory and the state space is small — every example in the core module — exact dynamic programming is the right tool, and it supplies the ground truth $V^{\star}$ and $\pi^{\star}$ that the Advanced notebooks use to measure their approximations. Rollout and MCTS become the practical choice when only a simulator is available or the state space is too large to sweep.
___


## Summary
This reading developed the theory and pseudocode for two approximate planners that need only simulated experience: random rollout for estimating a state's value under a policy, and Monte Carlo tree search for choosing an action at a root state.

> __Key takeaways:__
>
> 1. **Rollout estimates value by simulation:** the estimate $\hat V^{\pi}(s_{0})=\frac{1}{N}\sum_{i=1}^{N}G^{(i)}$ averages $N$ truncated trajectory returns. Its error has two parts: Monte Carlo noise with standard error $\sigma_{G}/\sqrt{N}$, controlled by $N$, and horizon-truncation bias, controlled by $H$.
> 2. **One-step rollout lookahead improves a policy:** an exact Bellman backup over rollout-estimated continuation values, $\hat Q(s,a)=R(s,a)+\gamma\sum_{s^{\prime}\in\mathcal{S}}P(s^{\prime}\mid s,a)\,\hat V^{\pi}(s^{\prime})$, yields an improved action at a state without solving the full MDP.
> 3. **MCTS searches from the root:** repeated selection (UCT), expansion, simulation, and backpropagation build the estimates $Q(s_{0},a)$, and the returned action $\arg\max_{a\in\mathcal{A}}Q(s_{0},a)$ converges to the optimal root action as the iteration budget grows, with the exploration constant $c$ balancing exploitation against exploration.

Rollout and MCTS trade the exactness and full-state coverage of dynamic programming for planners that need only a simulator and a budget of simulated trajectories. The companion Watch-Demo and Codio activity notebooks run both planners on the $5\times5$ grid world and measure them against the exact solution from value iteration.
___


### Additional Resources
* Sutton, R. S., & Barto, A. G. (2018). _Reinforcement Learning: An Introduction_ (2nd ed.), Chapter 8. MIT Press.
* Kochenderfer, M. J., Wheeler, T. A., & Wray, K. H. (2022). _Algorithms for Decision Making_. MIT Press.
* Bertsekas, D. P. (2020). _Rollout, Policy Iteration, and Distributed Reinforcement Learning_. Athena Scientific.
* Kocsis, L., & Szepesvári, C. (2006). Bandit Based Monte-Carlo Planning. In _European Conference on Machine Learning (ECML)_, pp. 282–293.
* Browne, C. B., Powley, E., Whitehouse, D., Lucas, S. M., Cowling, P. I., Rohlfshagen, P., Tavener, S., Perez, D., Samothrakis, S., & Colton, S. (2012). A Survey of Monte Carlo Tree Search Methods. _IEEE Transactions on Computational Intelligence and AI in Games_, 4(1), 1–43.